# Lending Club P2P Loan Default Prediction — Part 1: Baseline Reproduction

**UE24CS352A Machine Learning — Mini-Project, Team A-15 (Sec-A)**
Aman Priyansh (PES1UG24AM030) and Addanki Harsha Rajeswari (PES1UG24AM014)

Paper reproduced: *LendingClub Loan Default and Profitability Prediction*, Peiqian Li and Gao Han, Stanford CS229, 2018.

## 1. Data and the authors' code

In [ ]:
# CELL 1 - Get the data and the author's code
# 1) Download the Lending Club dataset from Kaggle (about 1.6 GB)
# 2) Download the paper's original code zip from the author's Google Drive
# We only READ the author's code to copy their steps. We do not run it.

import kagglehub, os, glob, zipfile, gdown

data_dir = kagglehub.dataset_download("wordsforthewise/lending-club")
print("Data folder:", data_dir)
for f in glob.glob(data_dir + "/**/*", recursive=True):
    print("  ", f, round(os.path.getsize(f)/1e6, 1), "MB")

gdown.download(id="1EilPoVCYx6fI84FUrfJvxGCUho5TImXx", output="author_code.zip", quiet=True)
with zipfile.ZipFile("author_code.zip") as z:
    z.extractall("author_code")
    print("Author files:", sorted(z.namelist()))

In [ ]:
# CELL 2 - Print the author's key code files so we can copy their exact steps
files = ["utils.py", "features_eng.py", "log_reg.py", "random_forest.py", "classification_eval.py",
         "utils_raw_return_regression.py", "ridge.py", "linear_regression.py", "mlp_regressor.py",
         "random_forest_regressor.py", "random_forest_shallow_regressor_10.py", "regression_eval.py", "regression_cost_eval.py"]
for name in files:
    print("=" * 30, name, "=" * 30)
    print(open("author_code/" + name).read())

## 2. Rebuild the missing cleaned dataset

In [ ]:
# CELL 3 - Rebuild the paper's "data_clean.csv"
# The author's code reads a file called data_clean.csv that is NOT in the zip.
# The paper (Section III-A) says how it was made:
#   - loans issued 2012 to 2015
#   - only loans with a FINAL status: "Fully Paid", "Charged Off", "Default"
#     (so "Current", "Late", "In Grace Period" are removed)
import pandas as pd, numpy as np

csv_file = data_dir + "/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"

# All columns the author's classification + regression code uses, plus issue_d to filter by year
author_cols = ['loan_amnt','funded_amnt','total_pymnt','term','int_rate','installment','issue_d','last_pymnt_d',
 'emp_length','home_ownership','annual_inc','verification_status','loan_status','purpose','zip_code','addr_state','dti',
 'delinq_2yrs','earliest_cr_line','fico_range_low','fico_range_high','inq_last_6mths','mths_since_last_delinq',
 'mths_since_last_record','open_acc','pub_rec','revol_bal','revol_util','total_acc','out_prncp','out_prncp_inv',
 'last_fico_range_high','last_fico_range_low','collections_12_mths_ex_med','mths_since_last_major_derog','policy_code',
 'application_type','annual_inc_joint','dti_joint','acc_now_delinq','tot_coll_amt','tot_cur_bal','open_acc_6m',
 'open_act_il','open_il_12m','open_il_24m','mths_since_rcnt_il','total_bal_il','il_util','open_rv_12m','open_rv_24m',
 'all_util','total_rev_hi_lim','inq_fi','total_cu_tl','inq_last_12m','acc_open_past_24mths','avg_cur_bal',
 'bc_open_to_buy','bc_util','chargeoff_within_12_mths','delinq_amnt','mo_sin_old_il_acct','mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl','mort_acc','mths_since_recent_bc','mths_since_recent_bc_dlq',
 'mths_since_recent_inq','mths_since_recent_revol_delinq','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
 'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats',
 'num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m','num_tl_op_past_12m','pct_tl_nvr_dlq','percent_bc_gt_75',
 'pub_rec_bankruptcies','tax_liens','tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit']

raw = pd.read_csv(csv_file, usecols=author_cols, low_memory=False)
print("Full Kaggle file:", raw.shape)

raw["issue_year"] = pd.to_datetime(raw["issue_d"], format="%b-%Y").dt.year
clean = raw[raw["issue_year"].between(2012, 2015) &
            raw["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
clean = clean.drop(columns=["issue_year"])
del raw

print("After filter (paper had 745,529):", len(clean))
print(clean["loan_status"].value_counts())
print((clean["loan_status"] != "Fully Paid").mean().round(3), "= share negative (paper: 0.19)")
clean.to_csv("data_clean.csv", index=False)

## 3. The authors' cleaning steps

In [ ]:
# CELL 4 - The author's cleaning steps (copied from their utils.py)
# Same three fill rules as the paper: mean-fill, zero-fill, and max-fill (max x 5),
# then one-hot encoding, a random 70/30 split, and standardisation fitted on train only.
#
# Changes ONLY so the 2018 code runs today (the maths is identical):
#   - int_rate / revol_util are already numbers in this file (e.g. 13.5), so we just divide by 100
#   - dates: "days since 1970" computed the modern way
#   - MEMORY: the author's pandas code makes several full copies of a 786k x 1,100 table.
#     Here we build ONE float32 table and standardise it in place, so it fits comfortably in RAM.
import pandas as pd, numpy as np, os, gc
if 'clean' not in globals():
    clean = pd.read_csv("data_clean.csv", low_memory=False)
print("Loans:", len(clean))

nan_max_cols = ['mths_since_last_delinq','mths_since_last_record','mths_since_last_major_derog','mths_since_rcnt_il',
 'mths_since_recent_bc','mths_since_recent_bc_dlq','mths_since_recent_inq','mths_since_recent_revol_delinq',
 'mo_sin_old_il_acct','mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl']
nan_zero_cols = ['tot_coll_amt','tot_cur_bal','open_acc_6m','open_act_il','open_il_12m','open_il_24m','total_bal_il',
 'open_rv_12m','open_rv_24m','inq_fi','inq_last_12m','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
 'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats',
 'num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m','num_tl_op_past_12m','mort_acc','percent_bc_gt_75',
 'tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit','bc_open_to_buy',
 'acc_open_past_24mths','total_cu_tl','total_rev_hi_lim']
nan_mean_cols = ['dti','annual_inc_joint','dti_joint','il_util','all_util','bc_util','pct_tl_nvr_dlq','avg_cur_bal','revol_util']

def author_clean(df):
    df = df.copy()
    df['revol_util'] = pd.to_numeric(df['revol_util'].astype(str).str.replace('%',''), errors='coerce') / 100.0
    df['int_rate']   = pd.to_numeric(df['int_rate'].astype(str).str.replace('%',''), errors='coerce') / 100.0
    for col in df.columns:
        if col in nan_mean_cols: df[col] = df[col].fillna(np.nanmean(df[col].values))
        if col in nan_zero_cols: df[col] = df[col].fillna(0.0)
        if col in nan_max_cols:  df[col] = df[col].fillna(np.nanmax(df[col].values) * 5)
    d = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
    df['earliest_cr_line'] = (d - pd.Timestamp('1970-01-01')).dt.days
    return df

def build_matrix(df, label_col, seed=0):
    # 1) shuffle rows (author: df.sample(frac=1)); seed fixed so we can re-run and get the same split
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    y = df[label_col].to_numpy(); df = df.drop(columns=[label_col])
    # 2) one-hot encoding (author: pd.get_dummies) - done straight into one float32 table
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    num_cols = [c for c in df.columns if c not in cat_cols]
    cats = {c: sorted(df[c].dropna().unique()) for c in cat_cols}
    names = num_cols + [c + "_" + str(v) for c in cat_cols for v in cats[c]]
    n, p = len(df), len(names)
    X = np.zeros((n, p), dtype=np.float32)
    X[:, :len(num_cols)] = df[num_cols].to_numpy(dtype=np.float32)
    j = len(num_cols)
    for c in cat_cols:
        codes = pd.Categorical(df[c], categories=cats[c]).codes.astype(np.int64)
        rows = np.where(codes >= 0)[0]
        X[rows, j + codes[rows]] = 1.0
        j += len(cats[c])
    print("Empty (NaN) cells left after the author's fills:", int(np.isnan(X).sum()))
    np.nan_to_num(X, copy=False)
    # 3) 70 / 30 split: first 70% of the shuffled rows = train (author: np.split at 0.7)
    n_tr = int(.7 * n)
    # 4) StandardScaler fitted on TRAIN only, applied to all rows, in place (mean 0, std 1)
    for k in range(0, p, 100):
        blk = X[:n_tr, k:k+100].astype(np.float64)
        m, s = blk.mean(0), blk.std(0); s[s == 0] = 1.0
        X[:, k:k+100] -= m.astype(np.float32); X[:, k:k+100] /= s.astype(np.float32)
    del df; gc.collect()
    return X[:n_tr], y[:n_tr], X[n_tr:], y[n_tr:], names

# ---- Classification data (author's prepare_data) ----
cls_df = author_clean(clean.drop(columns=['total_pymnt','issue_d','last_pymnt_d']))
cls_df['loan_status'] = cls_df['loan_status'].map({'Charged Off': 0, 'Default': 0, 'Fully Paid': 1})
train_x, train_y, test_x, test_y, cls_features = build_matrix(cls_df, 'loan_status')
del cls_df; gc.collect()
print("Train:", train_x.shape, " Test:", test_x.shape)
print("Number of features (paper: 1,097):", len(cls_features))
del clean; gc.collect()   # free 1 GB; data_clean.csv stays on disk for the regression part